In [1]:
import torch
import pytorch_lightning as pl
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
from collections import Counter
from sklearn.model_selection import train_test_split
import nltk

import torch.nn as nn
import torch.nn.functional as F
import torchmetrics


nltk.download('punkt', quiet=True)

True

In [2]:
pl.seed_everything(42)

INFO:lightning_fabric.utilities.seed:Seed set to 42


42

### 1. Поготовка данных

In [3]:
class HotelReviewDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=150):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        tokens = word_tokenize(text.lower())

        indices = [self.vocab.get(token, self.vocab['<UNK>']) for token in tokens]

        if len(indices) < self.max_len:
            indices += [self.vocab['<PAD>']] * (self.max_len - len(indices))
        else:
            indices = indices[:self.max_len]

        return torch.tensor(indices, dtype=torch.long), torch.tensor(label - 1, dtype=torch.long)

In [4]:
class HotelDataModule(pl.LightningDataModule):
    def __init__(self, csv_path, batch_size=64, max_len=150, min_freq=2):
        super().__init__()
        self.csv_path = csv_path
        self.batch_size = batch_size
        self.max_len = max_len
        self.min_freq = min_freq
        self.vocab = None

    def setup(self, stage=None):
        df = pd.read_csv(self.csv_path)

        train_val, test_df = train_test_split(df, test_size=0.2, stratify=df['Rating'], random_state=42)
        train_df, val_df = train_test_split(train_val, test_size=0.25, stratify=train_val['Rating'], random_state=42)

        if self.vocab is None:
            counter = Counter()
            for text in train_df['Review']:
                counter.update(word_tokenize(str(text).lower()))

            self.vocab = {'<PAD>': 0, '<UNK>': 1}
            idx = 2
            for word, count in counter.items():
                if count >= self.min_freq:
                    self.vocab[word] = idx
                    idx += 1

        self.train_ds = HotelReviewDataset(train_df['Review'].values, train_df['Rating'].values, self.vocab, self.max_len)
        self.val_ds = HotelReviewDataset(val_df['Review'].values, val_df['Rating'].values, self.vocab, self.max_len)
        self.test_ds = HotelReviewDataset(test_df['Review'].values, test_df['Rating'].values, self.vocab, self.max_len)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=2)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, num_workers=2)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, num_workers=2)


### 2. Базовый класс модели

Чтобы не дублировать код обучения для CNN и RNN, сделаем общий класс.

In [5]:
class BaseModel(pl.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.lr = lr
        self.criterion = nn.CrossEntropyLoss()
        self.accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=5)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = self.accuracy(logits, y)
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        acc = self.accuracy(logits, y)
        self.log('test_acc', acc)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


### 3. CNN

In [6]:
class CNNModel(BaseModel):
    def __init__(self, vocab_size, embed_dim=128, num_filters=128, kernel_size=3, num_classes=5, lr=1e-3):
        super().__init__(lr)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv = nn.Conv1d(embed_dim, num_filters, kernel_size)
        self.fc = nn.Linear(num_filters, num_classes)

    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)
        x = F.relu(self.conv(x))
        x = F.max_pool1d(x, x.size(2)).squeeze(-1)
        return self.fc(x)

### 4. LSTM

In [7]:
class LSTMModel(BaseModel):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=5, lr=1e-3):
        super().__init__(lr)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        return self.fc(hidden[-1])

In [8]:
dm = HotelDataModule('tripadvisor_hotel_reviews.csv')
dm.setup()

Building vocabulary...


In [9]:
cnn = CNNModel(vocab_size=len(dm.vocab))
trainer_cnn = pl.Trainer(max_epochs=5, accelerator='auto', devices=1)
trainer_cnn.fit(cnn, dm)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ criterion │ CrossEntropyLoss   │      0 │ train │     0 │
│ 1 │ accuracy  │ MulticlassAccuracy │      0 │ train │     0 │
│ 2 │ embedding │ Embedding          │  3.0 M │ train │     0 │
│ 3 │ conv      │ Conv1d             │ 49.3 K │ train │     0 │
│ 4 │ fc        │ Linear             │    645 │ train │     0 │
└───┴───────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 3.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.1 M                                                                                                
Total estimated model params size (MB): 12                                                                         
Modules in train mode: 5                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


In [10]:
trainer_cnn.test(cnn, dm)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.5794095993041992     │
└───────────────────────────┴───────────────────────────┘

[{'test_acc': 0.5794095993041992}]

In [11]:
lstm = LSTMModel(vocab_size=len(dm.vocab))
trainer_lstm = pl.Trainer(max_epochs=5, accelerator='auto', devices=1)
trainer_lstm.fit(lstm, dm)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ criterion │ CrossEntropyLoss   │      0 │ train │     0 │
│ 1 │ accuracy  │ MulticlassAccuracy │      0 │ train │     0 │
│ 2 │ embedding │ Embedding          │  3.0 M │ train │     0 │
│ 3 │ lstm      │ LSTM               │  132 K │ train │     0 │
│ 4 │ fc        │ Linear             │    645 │ train │     0 │
└───┴───────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 3.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.1 M                                                                                                
Total estimated model params size (MB): 12                                                                         
Modules in train mode: 5                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


In [12]:
trainer_lstm.test(lstm, dm)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │     0.439863383769989     │
└───────────────────────────┴───────────────────────────┘

[{'test_acc': 0.439863383769989}]